In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

processed_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "finGuide_deduplicated.json"
)

with open(processed_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total records: {len(data):,}")

Total records: 42,661


In [6]:
data[0]

{'id': 'FinQA_ADI/2009/page_49.pdf-1',
 'question': 'what is the the interest expense in 2009?',
 'answer': '380',
 'context': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .\nif libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .\nforeign currency exposure as more fully described in note 2i .\nin the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .\ndollar-based exposures by entering into forward foreign currency exchange contracts .\nthe terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .\ncurrently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .\nrelative

In [2]:
for record in data[:3]:
    print({
        "id": record["id"],
        "source_dataset": record["source_dataset"],
        "source_id": record["source_id"]
    })

{'id': 'FinQA_ADI/2009/page_49.pdf-1', 'source_dataset': 'FinQA', 'source_id': 'ADI/2009/page_49.pdf-1'}
{'id': 'FinQA_AAL/2018/page_13.pdf-2', 'source_dataset': 'FinQA', 'source_id': 'AAL/2018/page_13.pdf-2'}
{'id': 'FinQA_INTC/2013/page_71.pdf-4', 'source_dataset': 'FinQA', 'source_id': 'INTC/2013/page_71.pdf-4'}


In [4]:
def get_source_group(record):
    source_dataset = record["source_dataset"]
    source_id = record["source_id"]

    if source_dataset == "FinQA":
        # Remove the final question/example number
        # Example: ADI/2009/page_49.pdf-1
        #       → ADI/2009/page_49.pdf
        return source_id.rsplit("-", 1)[0]

    elif source_dataset == "FinQA-30K":
        # source_id = source_pdf + "_" + chunk_id
        # We need the PDF/document portion as the group.
        return source_id.rsplit("_", 1)[0]

    elif source_dataset == "ConvFinQA":
        # ConvFinQA source_id is based on the original document
        # and may contain a trailing conversation/example index.
        return source_id.rsplit("-", 1)[0]

    else:
        return source_id


for record in data[:10]:
    print(
        record["source_dataset"],
        "→",
        get_source_group(record)
    )

FinQA → ADI/2009/page_49.pdf
FinQA → AAL/2018/page_13.pdf
FinQA → INTC/2013/page_71.pdf
ConvFinQA → Single_ETR/2008/page_313.pdf
FinQA → C/2010/page_272.pdf
ConvFinQA → Single_AMT/2012/page_121.pdf
FinQA → GIS/2019/page_45.pdf
ConvFinQA → Single_IPG/2009/page_89.pdf
FinQA → CDNS/2018/page_32.pdf
FinQA → GIS/2008/page_83.pdf


In [5]:
count = 0

for record in data:
    if record["source_dataset"] == "FinQA-30K":
        print(
            "source_id:",
            record["source_id"],
            "→ group:",
            get_source_group(record)
        )
        count += 1

        if count == 10:
            break

source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf
source_id: 09_chapter 1.pdf_0 → group: 09_chapter 1.pdf


In [7]:
data[0].keys()

dict_keys(['id', 'question', 'answer', 'context', 'table', 'domain', 'question_type', 'reasoning', 'source_dataset', 'source_id'])

In [22]:
import random
from collections import defaultdict
import json

SEED = 42

# --------------------------------------------------
# 1. Create source groups
# --------------------------------------------------

groups = defaultdict(list)

for record in data:
    source_group = get_source_group(record)
    groups[source_group].append(record)


print(f"Total records : {len(data):,}")
print(f"Total groups  : {len(groups):,}")


# --------------------------------------------------
# 2. Shuffle groups reproducibly
# --------------------------------------------------

group_items = list(groups.items())

random.seed(SEED)
random.shuffle(group_items)


# --------------------------------------------------
# 3. Split groups into 80 / 10 / 10
# --------------------------------------------------

total_records = len(data)

train_target = total_records * 0.80
val_target = total_records * 0.10

train_data = []
val_data = []
test_data = []

train_count = 0
val_count = 0

for source_group, records in group_items:

    if train_count < train_target:
        train_data.extend(records)
        train_count += len(records)

    elif val_count < val_target:
        val_data.extend(records)
        val_count += len(records)

    else:
        test_data.extend(records)


# --------------------------------------------------
# 4. Shuffle records inside each split
# --------------------------------------------------

random.seed(SEED)

random.shuffle(train_data)
random.shuffle(val_data)
random.shuffle(test_data)


# --------------------------------------------------
# 5. Print results
# --------------------------------------------------

print("\nSplit results:")
print(f"Train      : {len(train_data):,} ({len(train_data)/total_records:.2%})")
print(f"Validation : {len(val_data):,} ({len(val_data)/total_records:.2%})")
print(f"Test       : {len(test_data):,} ({len(test_data)/total_records:.2%})")

print("\nTotal:")
print(len(train_data) + len(val_data) + len(test_data))

Total records : 42,661
Total groups  : 3,958

Split results:
Train      : 34,185 (80.13%)
Validation : 4,268 (10.00%)
Test       : 4,208 (9.86%)

Total:
42661


In [23]:
len(group_items)

3958

In [24]:
# --------------------------------------------------
# Verify no source-group leakage
# --------------------------------------------------

train_groups = {
    get_source_group(record)
    for record in train_data
}

val_groups = {
    get_source_group(record)
    for record in val_data
}

test_groups = {
    get_source_group(record)
    for record in test_data
}


train_val_overlap = train_groups & val_groups
train_test_overlap = train_groups & test_groups
val_test_overlap = val_groups & test_groups


print("Source-group overlap:")
print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test      :", len(train_test_overlap))
print("Validation ∩ Test :", len(val_test_overlap))


if not train_val_overlap and not train_test_overlap and not val_test_overlap:
    print("\n✅ LEAKAGE CHECK PASSED")
else:
    print("\n❌ LEAKAGE DETECTED")

Source-group overlap:
Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0

✅ LEAKAGE CHECK PASSED


In [27]:
import json
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
# Output paths
train_path = PROCESSED_DATA_DIR / "train.json"
validation_path = PROCESSED_DATA_DIR / "validation.json"
test_path = PROCESSED_DATA_DIR / "test.json"


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )


# Save splits
save_json(train_data, train_path)
save_json(val_data, validation_path)
save_json(test_data, test_path)


print("Saved successfully:")
print(f"Train      : {train_path}")
print(f"Validation : {validation_path}")
print(f"Test       : {test_path}")

Saved successfully:
Train      : d:\ai_project\FinGuide-AI\data\processed\train.json
Validation : d:\ai_project\FinGuide-AI\data\processed\validation.json
Test       : d:\ai_project\FinGuide-AI\data\processed\test.json


In [28]:
# Reload and verify
with open(train_path, "r", encoding="utf-8") as f:
    train_check = json.load(f)

with open(validation_path, "r", encoding="utf-8") as f:
    validation_check = json.load(f)

with open(test_path, "r", encoding="utf-8") as f:
    test_check = json.load(f)


print("Reload verification:")
print(f"Train      : {len(train_check):,}")
print(f"Validation : {len(validation_check):,}")
print(f"Test       : {len(test_check):,}")

total_check = (
    len(train_check)
    + len(validation_check)
    + len(test_check)
)

print(f"Total      : {total_check:,}")

assert total_check == 42661
assert len(train_check) == 34185
assert len(validation_check) == 4268
assert len(test_check) == 4208

print("\n✅ SPLIT FILES VERIFIED")

Reload verification:
Train      : 34,185
Validation : 4,268
Test       : 4,208
Total      : 42,661

✅ SPLIT FILES VERIFIED
